In [1]:
import os
import shutil
import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from facenet_pytorch import MTCNN
import cv2
import glob
from tqdm import tqdm
import re

/home/ryuko/Documents/Codes/Python/Skripsi/Convat-1st/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
# --- Configuration ---
# Ensure this base path points to your 'testing_output' directory
BASE_INPUT_DIR = '../Casme3/'
SUBJECT_ID = 'color'
SUBJECT_CATEGORY = 'a'
SUBJECT_DIR = os.path.join(BASE_INPUT_DIR, SUBJECT_CATEGORY, SUBJECT_ID)

# --- Output Paths ---
ANALYSIS_OUTPUT_DIR = f'./analysis_output_{SUBJECT_ID}'
os.makedirs(ANALYSIS_OUTPUT_DIR, exist_ok=True)

# Define specific file and directory paths for outputs
EXCEL_OUTPUT_PATH = os.path.join(ANALYSIS_OUTPUT_DIR, f'landmark_displacement_{SUBJECT_ID}.xlsx')
PLOT_DISPLACEMENT_PATH = os.path.join(ANALYSIS_OUTPUT_DIR, f'displacement_plot_{SUBJECT_ID}.png')
VISUAL_COMPARE_DIR = os.path.join(ANALYSIS_OUTPUT_DIR, f'frame_comparisons_{SUBJECT_ID}')
PIXEL_ANALYSYS_DIR = os.path.join(ANALYSIS_OUTPUT_DIR, f'pixel_difference_analysis_{SUBJECT_ID}')

# Create sub-directories for visual outputs
os.makedirs(VISUAL_COMPARE_DIR, exist_ok=True)
os.makedirs(PIXEL_ANALYSYS_DIR, exist_ok=True)

RESIZE_DIMENSIONS = (256, 256)

print(f"Input Directory set to: {SUBJECT_DIR}")
print(f"Output Directory set to: {ANALYSIS_OUTPUT_DIR}")

Input Directory set to: ../Casme3/a/color
Output Directory set to: ./analysis_output_color


In [22]:
def setup_detector():
    """Initializes the MTCNN face detector."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    print("Setting up MTCNN face detector... (This may take a moment on first run)")
    # keep_all=True detects all faces, though we'll focus on the first
    face_detector = MTCNN(keep_all=True, device=device)
    return face_detector

mtcnn = setup_detector()
print("MTCNN detector is ready.")

Using device: cuda
Setting up MTCNN face detector... (This may take a moment on first run)
MTCNN detector is ready.


In [23]:
def natural_sort_key(filename):
    basename = os.path.basename(filename)
    parts = re.findall(r'frame_second_(\d+)_frame_(\d+)', basename)
    if parts:
        second = int(parts[0][0])
        frame = int(parts[0][1])
        return (second, frame)
    else:
        return (float('inf'), float('inf'))

In [24]:
def get_landmarks(img_pil, face_detector):
    """
    Detects face and 5-point landmarks from a PIL Image.

    Args:
        img_pil (PIL.Image): The input image.
        face_detector (MTCNN): The initialized MTCNN detector.

    Returns:
        np.ndarray: A (5, 2) array of landmark coordinates, or None if no face is found.
    """
    try:
        boxes, probs, landmarks = face_detector.detect(img_pil, landmarks=True)
        # If no landmarks are detected, return None
        if landmarks is None or len(landmarks) == 0:
            return None
        # Assume the first detected face is the primary subject
        return landmarks[0]
    except Exception as e:
        print(f"Error during landmark detection: {e}")
        return None

def calculate_displacement(points1, points2):
    """
    Calculates the Euclidean distance between two sets of 5-point landmarks.

    Args:
        points1 (np.ndarray): The first set of (5, 2) landmarks (reference).
        points2 (np.ndarray): The second set of (5, 2) landmarks (current).

    Returns:
        tuple:
            - list: A list of 5 distances, one for each landmark point.
            - float: The mean displacement of all 5 points.
    """
    if points1 is None or points2 is None:
        return [np.nan] * 5, np.nan

    distances = []
    for i in range(len(points1)):
        # Calculate Euclidean distance: sqrt((x2-x1)^2 + (y2-y1)^2)
        dist = np.linalg.norm(points1[i] - points2[i])
        distances.append(dist)

    mean_displacement = np.nanmean(distances)
    return distances, mean_displacement

def draw_landmarks_on_image(img_cv2, landmarks, color_bgr):
    """
    Draws 5 landmark points onto a cv2 (BGR) image.

    Args:
        img_cv2 (np.ndarray): The image in OpenCV BGR format.
        landmarks (np.ndarray): The (5, 2) array of landmarks to draw.
        color_bgr (tuple): The (B, G, R) color for the points.

    Returns:
        np.ndarray: The image with landmarks drawn on it.
    """
    if landmarks is None:
        return img_cv2

    img_with_landmarks = img_cv2.copy()
    for (x, y) in landmarks:
        cv2.circle(img_with_landmarks, (int(x), int(y)), 3, color_bgr, -1) # -1 fills the circle
    return img_with_landmarks

print("Helper functions defined.")

Helper functions defined.


In [38]:
print(f"Starting analysis for subject in: {SUBJECT_DIR}")

image_files_list = sorted(glob.glob(os.path.join(SUBJECT_DIR, '*.jpg')))
image_files = sorted(image_files_list, key=natural_sort_key)

if not image_files:
    print(f"WARNING: No .jpg images found in {SUBJECT_DIR}.")
    all_displacement_data = []
    reference_image_path = None
    reference_landmarks = None
else:
    print(f"Found {len(image_files)} image frames.")

    all_displacement_data = []
    reference_landmarks = None
    reference_image_path = None

    if image_files:
        reference_image_path = image_files[0]
        try:
            img_pil_ref = Image.open(reference_image_path).convert('RGB')
            reference_landmarks = get_landmarks(img_pil_ref, mtcnn)
            if reference_landmarks is None:
                print(f"Warning: No face detected in reference frame {reference_image_path}")
        except Exception as e:
            print(f"Error opening/processing reference frame {reference_image_path}: {e}")

    for img_path in tqdm(image_files, desc="Calculating Landmark Displacement"):
        frame_filename = os.path.basename(img_path)

        if reference_landmarks is None:
            all_displacement_data.append([frame_filename] + [np.nan] * 5 + [np.nan])
            continue

        try:
            img_pil_curr = Image.open(img_path).convert('RGB')
            current_landmarks = get_landmarks(img_pil_curr, mtcnn)

            distances, mean_dist = calculate_displacement(reference_landmarks, current_landmarks)
            all_displacement_data.append([frame_filename] + distances + [mean_dist])

        except Exception as e:
            print(f"Error processing {frame_filename}: {e}")
            all_displacement_data.append([frame_filename] + [np.nan] * 5 + [np.nan])

print("Displacement calculation complete.")

Starting analysis for subject in: ../Casme3/a/color
Found 836 image frames.


Calculating Landmark Displacement: 100%|██████████| 836/836 [00:23<00:00, 35.74it/s]

Displacement calculation complete.


In [42]:
if not all_displacement_data:
    print("No data was processed. Excel file will not be created.")
    df_displacement = pd.DataFrame()
else:
    columns = ['frame',
               'landmark_0_dist (L-Eye)', 'landmark_1_dist (R-Eye)',
               'landmark_2_dist (Nose)',
               'landmark_3_dist (L-Mouth)', 'landmark_4_dist (R-Mouth)',
               'mean_displacement']

    df_displacement = pd.DataFrame(all_displacement_data, columns=columns)

    df_displacement.to_excel(EXCEL_OUTPUT_PATH, index=False)
    print(f"\nSuccessfully saved displacement data to: {EXCEL_OUTPUT_PATH}")

    print("\n--- Displacement Data (Head) ---")
    print(df_displacement.head())

    print("\n--- Descriptive Statistics (Mean Displacement) ---")
    print(df_displacement['mean_displacement'].describe())


Successfully saved displacement data to: ./analysis_output_color/landmark_displacement_color.xlsx

--- Displacement Data (Head) ---
     frame  landmark_0_dist (L-Eye)  landmark_1_dist (R-Eye)  \
0   10.jpg                 0.000000                 0.000000   
1  100.jpg                 8.890659                 4.741364   
2  101.jpg                 8.890659                 4.741364   
3  102.jpg                 9.891868                 5.200109   
4  103.jpg                 8.538896                 5.061815   

   landmark_2_dist (Nose)  landmark_3_dist (L-Mouth)  \
0                0.000000                   0.000000   
1                9.843283                   6.142409   
2                9.843283                   6.142409   
3               10.722733                   6.324989   
4                9.599830                   6.153076   

   landmark_4_dist (R-Mouth)  mean_displacement  
0                   0.000000           0.000000  
1                   4.411634           6.8058

In [43]:
if df_displacement.empty or 'mean_displacement' not in df_displacement.columns:
    print("DataFrame is empty. Skipping displacement plot generation.")
else:
    print("\nGenerating displacement plot...")
    plt.figure(figsize=(15, 7))

    plt.plot(df_displacement.index, df_displacement['mean_displacement'],
             label='Mean Displacement', color='red', linewidth=2)

    plt.xticks(df_displacement.index[::20])
    plt.xlim(df_displacement.index.min(), df_displacement.index.max())

    plt.ylim(df_displacement['mean_displacement'].min(), df_displacement['mean_displacement'].max())

    plt.title(f'Landmark Displacement for Subject {SUBJECT_ID} (Relative to First Frame)')
    plt.xlabel('Frame Index')
    plt.ylabel('Displacement (pixels)')
    plt.legend(loc='upper right')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()

    plt.savefig(PLOT_DISPLACEMENT_PATH)
    print(f"Displacement plot saved to: {PLOT_DISPLACEMENT_PATH}")
    plt.close()



Generating displacement plot...
Displacement plot saved to: ./analysis_output_color/displacement_plot_color.png


In [44]:
if df_displacement.empty or not reference_image_path or reference_landmarks is None:
    print("Data, reference path, or reference_landmarks missing. Skipping frame comparison.")
else:
    print("\nGenerating landmark comparison visualizations...")

    idx_to_visualize = []
    if not df_displacement.empty:
        idx_to_visualize.append(0)
        if len(df_displacement) > 2:
            idx_to_visualize.append(len(df_displacement) // 2)

        max_dist_idx = df_displacement['mean_displacement'].idxmax()
        if max_dist_idx not in idx_to_visualize:
            idx_to_visualize.append(max_dist_idx)

    try:
        landmarks_ref = reference_landmarks

        for idx in idx_to_visualize:
            try:
                current_img_path = image_files[idx]
                current_filename = os.path.basename(current_img_path)

                img_pil_curr = Image.open(current_img_path).convert('RGB')
                landmarks_curr = get_landmarks(img_pil_curr, mtcnn)

                img_cv2_curr = cv2.cvtColor(np.array(img_pil_curr), cv2.COLOR_RGB2BGR)

                img_cv2_compare = draw_landmarks_on_image(img_cv2_curr, landmarks_ref, color_bgr=(255, 0, 0))
                img_cv2_compare = draw_landmarks_on_image(img_cv2_compare, landmarks_curr, color_bgr=(0, 0, 255))

                output_filename = os.path.join(VISUAL_COMPARE_DIR, f'compare_frame_{idx}_{current_filename}')
                cv2.imwrite(output_filename, img_cv2_compare)

            except Exception as e:
                print(f"Error visualizing frame {idx}: {e}")

        print(f"Comparison visualizations saved to: {VISUAL_COMPARE_DIR}")

    except Exception as e:
        print(f"Error during comparison: {e}")


Generating landmark comparison visualizations...
Comparison visualizations saved to: ./analysis_output_color/frame_comparisons_color


In [45]:
if df_displacement.empty or not reference_image_path:
    print("DataFrame or reference image path is missing. Skipping pixel difference analysis.")
else:
    print("\nPerforming pixel difference analysis...")
    try:
        idx_max_dist = df_displacement['mean_displacement'].idxmax()
        img_path_max_dist = image_files[idx_max_dist]

        img_pil_ref = Image.open(reference_image_path).convert('RGB').resize(RESIZE_DIMENSIONS)
        img_pil_max = Image.open(img_path_max_dist).convert('RGB').resize(RESIZE_DIMENSIONS)

        gray_ref = np.array(img_pil_ref.convert('L'))
        gray_max = np.array(img_pil_max.convert('L'))

        diff_image = cv2.absdiff(gray_ref, gray_max)
        _, thresh_diff = cv2.threshold(diff_image, 30, 255, cv2.THRESH_BINARY)

        diff_path = os.path.join(PIXEL_ANALYSYS_DIR, 'pixel_diff_ref_vs_max.png')
        cv2.imwrite(diff_path, thresh_diff)
        print(f"Pixel difference (Ref vs. Max) image saved to: {diff_path}")

        blank_image = np.zeros(RESIZE_DIMENSIONS, dtype=np.uint8)
        diff_with_blank = cv2.absdiff(gray_ref, blank_image)

        blank_diff_path = os.path.join(PIXEL_ANALYSYS_DIR, 'pixel_diff_ref_vs_blank.png')
        cv2.imwrite(blank_diff_path, diff_with_blank)
        print(f"Pixel difference (Ref vs. Blank) image saved to: {blank_diff_path}")

    except Exception as e:
        print(f"Error during pixel analysis: {e}")

print("\n--- Full Analysis Complete ---")


Performing pixel difference analysis...
Pixel difference (Ref vs. Max) image saved to: ./analysis_output_color/pixel_difference_analysis_color/pixel_diff_ref_vs_max.png
Pixel difference (Ref vs. Blank) image saved to: ./analysis_output_color/pixel_difference_analysis_color/pixel_diff_ref_vs_blank.png

--- Full Analysis Complete ---
